# Question generator for failure modes
Last update: _2026-11-16_

## Preprocessing

### Loading extracted chapters

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
def load_txt_files_to_dataframe(folder_path: str | Path) -> pd.DataFrame:
    """
    Read all *.txt files from a folder into a pandas DataFrame.
    
    Parameters
    ----------
    folder_path : str or Path
        Path to the folder containing the .txt files.
        
    Returns
    -------
    pd.DataFrame
        DataFrame with two columns:
        - chapter : filename without the .txt extension
        - content : full text content of the file
    """
    folder = Path(folder_path)
    
    if not folder.is_dir():
        raise NotADirectoryError(f"Folder not found: {folder}")
    
    records = []
    
    for txt_file in sorted(folder.glob("*.txt")):
        chapter = txt_file.stem          # filename without extension
        content = txt_file.read_text(encoding="utf-8")
        records.append({
            "chapter": chapter,
            "content": content
        })
    
    df = pd.DataFrame(records, columns=["chapter", "content"])
    return df

In [3]:
folder = "output"
df = load_txt_files_to_dataframe(folder)

In [4]:
df['content'] = (
    df['content']
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.replace("\t", " ", regex=False)
)

In [5]:
df[:5]

,chapter,content
0,chapter_1,"32 Preface Mahmoud Elhussini, Montclair State ..."
1,chapter_10,Chapter 9 Achieving Operational Excellence an...
2,chapter_11,"Chapter 10 E-commerce: Digital Markets, Digit..."
3,chapter_12,Chapter 11 Managing Knowledge and Artificial ...
4,chapter_13,This page is intentionally left blank 522 ...


### Content splitter

In [6]:
from typing import Any

In [7]:
def split_contents(
    df: pd.DataFrame,
    text_splitter: Any,
    chapter_column: str = "chapter",
    content_column: str = "content",
) -> pd.DataFrame:
    """
    Split the text content of a DataFrame using a LangChain text splitter.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame that must contain at least the columns
        specified by `chapter_column` and `content_column`.
    text_splitter : Any
        An initialized LangChain text splitter instance
        (e.g. RecursiveCharacterTextSplitter, CharacterTextSplitter, ...).
        It must expose a `.split_text(text: str) -> list[str]` method.
    content_column : str, default "content"
        Name of the column that holds the raw text to be split.
    chapter_column : str, default "chapter"
        Name of the column that holds the chapter / source identifier.
        This value is copied to every resulting chunk.

    Returns
    -------
    pd.DataFrame
        New DataFrame with two columns:
        - chapter : original chapter name (repeated for each chunk)
        - chunks : the individual text chunks produced by the splitter
    """
    if chapter_column not in df.columns:
        raise ValueError(f"Column '{chapter_column}' not found in DataFrame")
    if content_column not in df.columns:
        raise ValueError(f"Column '{content_column}' not found in DataFrame")

    records = []

    for _, row in df.iterrows():
        chapter = row[chapter_column]
        text = row[content_column]

        # Guard against empty / NaN content
        if pd.isna(text) or not str(text).strip():
            continue

        chunks = text_splitter.split_text(str(text))

        for chunk in chunks:
            records.append({
                chapter_column: chapter,
                'chunks': chunk,
            })

    result_df = pd.DataFrame(records, columns=[chapter_column, 'chunks'])
    return result_df

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=[".\n", ". ", "!", "?"],
    chunk_size=500,  # Adjust size to roughly fit 2-3 sentences
    chunk_overlap=0,
)

In [9]:
chunks_df = split_contents(df, splitter)

In [10]:
chunks_df['source'] = 'Laudon_MIS_17th'

In [11]:
chunks_df.shape

(1675, 3)

In [12]:
chunks_df[:5]

,chapter,chunks,source
0,chapter_1,"32 Preface Mahmoud Elhussini, Montclair State ...",Laudon_MIS_17th
1,chapter_1,". Kumarage, Saint Cloud State University Weiqi...",Laudon_MIS_17th
2,chapter_1,". Michael Raisinghani, Texas Woman’s Universit...",Laudon_MIS_17th
3,chapter_1,. QuickEats utilizes AWM Frictionless and feat...,Laudon_MIS_17th
4,chapter_1,". For example, if a customer picks up a box o...",Laudon_MIS_17th


In [13]:
chunks_df.loc[2, 'chunks']

'. Michael Raisinghani, Texas Woman’s University Patricia Ryan, Southeast Missouri State University Ethné Swartz, Montclair State University Amir Talaei-Khoei, University of Nevada Reno Paulus Van Vliet, University of Nebraska at Omaha Global Edition Acknowledgments Contributors Muhammad Shakaib Akram, King Saud University Saadat Alhashmi, University of Sharjah Subhashini Bhaskaran, Ahlia University Bernard Bouwman Joyce Chan, City University of Hong Kong June Clarke, Sheffield Hallam University Paul Griffin, Singapore Management University Imran Medi, Asia Pacific University of Technology and Innovation Chandra Reka Ramachandiran, Asia Pacific University of Technology and Innovation Yasser Mohamed Ibrahim Sabri, Cairo University Bernd Schenk, University of Liechtenstein Neerja Sethi, Nanyang Technological University Vijay Sethi, Nanyang Technological University Jon and Diane Sutherland Carolien van den Berg, University of the Western Cape Reviewers Zach Lee, Durham University Nash Mil

The input CSV should be in the following format:
- source: name of the book
- chunks: str: Chunks of the preprocessed input text
- chapter: str: Subunit within book/unit

In [16]:
from src.Helper import Log

In [19]:
with Log.save_file("output/chunks_{:}.csv".format(chunks_df.loc[0, 'source'])) as filename:
    chunks_df[['source', 'chapter', 'chunks']].to_csv(filename, index=False)

File saved to `output\chunks_Laudon_MIS_17th_202611160910.csv` (727.1KB)


## Reconciled failure modes sampling

| Requires question generation | Failure Mode | Pipeline Stage / Part Checked | Description | Paper |
| --: | --- | --- | --- | --- |
| no | FP1 – Missing Content | Content / Knowledge Base | Question cannot be answered from the available documents; system may still produce an answer instead of abstaining. | Barnett et al. |
| no | FP2 – Missed the Top Ranked Documents | Retrieval | Answer exists in a document but the document is ranked too low to be returned in the top-k. | Barnett et al. |
| no | FP3 – Not in Context | Context Construction / Consolidation | Relevant documents were retrieved but were dropped during context consolidation (token limits, ranking, etc.). | Barnett et al. |
| no | FP4 – Not Extracted | Generation (LLM) | Answer is present in the provided context, but the LLM fails to extract it (often due to noise or contradictions). | Barnett et al. |
| mostly no | FP5 – Wrong Format | Generation | Question requests a specific format (table, list, etc.) but the LLM ignores the instruction. | Barnett et al. |
| mostly no | FP6 – Incorrect Specificity | Generation | Answer is returned but is either too general or too specific relative to the user’s need. | Barnett et al. |
| no | FP7 – Incomplete | Generation | Answer is not incorrect but omits information that was present in the context. | Barnett et al. |
| no | Noise Robustness | Retrieval + Generation | Ability to answer correctly when retrieved documents contain noise/irrelevant content. | Chen et al. (RGB) |
| no | Negative Rejection | Generation | Ability to refuse to answer (or say “I don’t know”) when no relevant information is present. | Chen et al. (RGB) |
| yes | Information Integration | Generation | Ability to combine information scattered across multiple retrieved documents. | Chen et al. (RGB) |
| no | Counterfactual Robustness | Generation | Ability to detect and resist factually incorrect / counterfactual information in the retrieved context. | Chen et al. (RGB) |
| no | E1 – Overchunking | Chunking | Documents split into excessively small/disjointed segments, losing context. | Leung et al. |
| no | E2 – Underchunking | Chunking | Chunks too large, mixing multiple topics and diluting relevant information. | Leung et al. |
| no | E3 – Context Mismatch | Chunking | Arbitrary split points break logical/semantic links (e.g., definition separated from its subject). | Leung et al. |
| no | E4 – Missed Retrieval | Retrieval | Relevant chunks are not retrieved at all. | Leung et al. |
| no | E5 – Low Relevance | Retrieval | Retrieved chunks have low topical relevance to the query. | Leung et al. |
| mostly no | E6 – Semantic Drift | Retrieval | Retrieved content drifts away from the original query intent. | Leung et al. |
| no | E7 – Low Recall | Reranking | Relevant items are ranked too low or filtered out by the reranker. | Leung et al. |
| no | E8 – Low Precision | Reranking | Irrelevant items are ranked highly by the reranker. | Leung et al. |
| no | E9 – Abstention Failure | Generation | Model answers incorrectly instead of abstaining when it should. | Leung et al. |
| no | E10 – Fabricated Content | Generation | Model invents information not present in the retrieved context (hallucination). | Leung et al. |
| no | E11 – Parametric Overreliance | Generation | Model ignores retrieved context and relies on its internal parametric knowledge. | Leung et al. |
| no | E12 – Incomplete Answer | Generation | Correct information is present but the answer is incomplete. | Leung et al. |
| mostly no | E13 – Misinterpretation | Generation | Model misuses or misreads the retrieved context. | Leung et al. |
| mostly no | E14 – Contextual Misalignment | Generation | Response is factually correct but off-topic relative to the query. | Leung et al. |
| yes | E15 – Chronological Inconsistency | Generation | Temporal order or dates are incorrect. | Leung et al. |
| yes | E16 – Numerical Error | Generation | Incorrect calculations or numerical representations. | Leung et al. |
| no | F1 – Outdated/Stale Data | Ingestion | Knowledge base contains outdated documents; answers rely on obsolete information. | Garani |
| no | F2 – Missing/Incomplete Documents | Ingestion | Required documents are absent or incomplete in the corpus. | Garani |
| no | F3 – Layout Parsing Errors | Ingestion | Errors in parsing document layout (tables, columns, etc.) corrupt content. | Garani |
| mostly no | F4 – Multimodality Conversion Loss | Ingestion | Information loss when converting non-text modalities (images, tables, etc.). | Garani |
| no | F5 – Tokenization Fragmentation | Representation | Tokenization splits domain-specific terms or meaningful units. | Garani |
| no | F6 – Embedding Drift / Model Mismatch | Representation | Embeddings become outdated or mismatch between indexing and query models. | Garani |
| no | F7 – Chunking Boundary Errors | Retrieval | Poor chunk boundaries hide or split the answer. | Garani |
| no | F8 – Domain Embedding Mismatch | Retrieval | General-purpose embeddings fail on domain-specific terminology. | Garani |
| yes | F9 – Multi-Hop Reasoning Gaps | Retrieval | System fails to retrieve the chain of documents needed for multi-hop questions. | Garani |
| mostly no | F10 – Graph RAG Trade-offs | Retrieval | Trade-offs inherent to graph-based retrieval (coverage vs. precision, etc.). | Garani |
| no | F11 – Low Recall / Ranking Failures | Retrieval | Relevant documents ranked too low or not retrieved. | Garani |
| no | F12 – Position-of-Gold Bias | Retrieval | “Lost-in-the-middle” effect; relevant content in the middle of the context is ignored. | Garani |
| no | F13 – Hallucination Despite Context | Generation | Model hallucinates even when correct information is present in context. | Garani |
| no | F14 – Conflicting Info Unresolved | Generation | Contradictory information in context is not resolved or flagged. | Garani |
| no | F15 – Incomplete / Partial Answers | Generation | Answers omit available relevant information. | Garani |
| no | F16 – Incorrect Specificity | Generation | Answer is too broad or too narrow for the user’s need. | Garani |
| no | F17 – Wrong Output Format | Generation | Model ignores requested output format. | Garani |
| no | F18 – Metric Inadequacy | Evaluation | Evaluation metrics fail to capture the relevant failure modes. | Garani |
| no | F19 – Lack of Continuous Monitoring | Evaluation | No ongoing production monitoring; quality degrades silently. | Garani |
| no | F20 – Latency / Cost Bottlenecks | Deployment | Excessive latency or cost makes the system impractical. | Garani |
| no | F21 – Auditability Gaps | Deployment | Difficulty tracing how an answer was produced (sources, ranking, etc.). | Garani |
| no | F22 – Domain Portability Degradation | Deployment | Performance drops sharply when moving to a new domain. | Garani |
| mostly no | F23 – PII / Compliance Leaks | Deployment | Sensitive or private information is exposed in answers. | Garani |
| no | F24 – Prompt Sensitivity | Deployment | Small prompt changes cause large performance swings. | Garani |
| no | F25 – Authorization / Policy Failures | Deployment | Access-control or policy rules are not enforced. | Garani |
| yes | F26 – Planning Failures | Agentic Orchestration | Agent fails to produce a correct plan or decomposition. | Garani |
| mostly no | F27 – Tool Selection / Execution Errors | Agentic Orchestration | Wrong tool chosen or tool call fails. | Garani |
| mostly no | F28 – Context Memory Degradation | Agentic Orchestration | Long-running agent loses or corrupts earlier context. | Garani |
| yes | F29 – Multi-Agent Coordination Failures | Agentic Orchestration | Multiple agents fail to coordinate or hand off correctly. | Garani |
| mostly no | F30 – Recursive Hallucination Cascades | Agentic Orchestration | Hallucinated intermediate results trigger further hallucinated queries. | Garani |
| no | F31 – Unbounded Cost / Latency Spirals | Agentic Orchestration | Agent loops cause uncontrolled cost or latency growth. | Garani |
| mostly no | F32 – Unsafe Reasoning Chains | Agentic Orchestration | Reasoning path produces unsafe or policy-violating conclusions. | Garani |
| no | F33 – Attribution & Governance Gaps | Agentic Orchestration | Inability to attribute decisions or enforce governance in multi-step agent runs. | Garani |

### Markdown to Pandas dataframe

In [ ]:
import re
from io import StringIO

In [ ]:
def extract_markdown_tables_from_notebook(nb_path: str):
    """Return a list of Markdown tables found in a .ipynb file."""
    import json
    with open(nb_path, encoding="utf-8") as f:
        nb = json.load(f)

    tables = []
    for cell in nb.get("cells", []):
        if cell.get("cell_type") == "markdown":
            source = "".join(cell.get("source", []))
            # very simple heuristic: look for lines that start with |
            if re.search(r"^\|.*\|", source, re.MULTILINE):
                tables.append(source)
    return tables

def markdown_table_to_df(md_table: str, header: bool = True) -> pd.DataFrame:
    """
    Convert a Markdown table string into a Pandas DataFrame.

    Parameters
    ----------
    md_table : str
        The full Markdown table (including the header and separator lines).
    header : bool, default True
        Whether the first row should be treated as column names.

    Returns
    -------
    pd.DataFrame
    """
    # Remove leading/trailing whitespace and empty lines
    lines = [line.strip() for line in md_table.strip().splitlines() if line.strip()]

    if len(lines) < 2:
        raise ValueError("Markdown table must contain at least a header and a separator line.")

    # Detect and drop the separator line (the one that looks like |---|---|)
    sep_pattern = re.compile(r"^\|?\s*:?-+:?\s*(\|\s*:?-+:?\s*)+\|?$")
    cleaned = []
    for line in lines:
        if sep_pattern.match(line):
            continue
        cleaned.append(line)

    if not cleaned:
        raise ValueError("No data rows found after removing the separator.")

    # Normalize pipes and turn the table into CSV-like text
    csv_lines = []
    for line in cleaned:
        # Strip outer pipes and split on remaining pipes
        cells = [cell.strip() for cell in line.strip("|").split("|")]
        csv_lines.append(",".join(f'"{c}"' for c in cells))  # quote to protect commas

    csv_text = "\n".join(csv_lines)

    # Read with pandas
    df = pd.read_csv(StringIO(csv_text), header=0 if header else None)

    return df

In [ ]:
tables = extract_markdown_tables_from_notebook("test_set_generator.ipynb")
pdf_failure_modes = markdown_table_to_df(tables[0])

In [ ]:
pdf_failure_modes['Pipeline Stage / Part Checked'].value_counts()

In [ ]:
pdf_failure_modes['Requires question generation'].value_counts()

In [ ]:
pdf_failure_modes.loc[1, 'Failure Mode'], pdf_failure_modes.loc[1, 'Description']

In [ ]:
pdf_failure_modes[pdf_failure_modes['Requires question generation'] == "yes"][['Failure Mode', 'Pipeline Stage / Part Checked', 'Description', 'Requires question generation', 'Paper']]

## Content sampling

### Helper functions

In [ ]:
import math
import numpy as np

In [ ]:
def sample_stratified(df: pd.DataFrame, num_entries: int, stratification_base) -> pd.DataFrame:
    """
    Perform stratified sampling on a DataFrame.
    
    Parameters
    ----------
    df : pd.DataFrame
        The input DataFrame.
    num_entries : int
        Desired number of samples (total size of the returned sample).
    stratification_base : str or list of str
        Column name (or list of column names) used to define the strata.
        
    Returns
    -------
    pd.DataFrame
        A stratified sample of approximately `n` rows.
    """
    if not isinstance(stratification_base, (str, list, tuple)):
        raise TypeError("stratification_base must be a column name (str) or a list of column names")
    
    if isinstance(stratification_base, str):
        stratification_base = [stratification_base]
    
    # Check that the stratification columns exist
    missing = [col for col in stratification_base if col not in df.columns]
    if missing:
        raise KeyError(f"Column(s) not found in DataFrame: {missing}")
    
    if num_entries <= 0:
        raise ValueError("n must be a positive integer")
    
    if num_entries >= len(df):
        return df.copy()
    
    # Compute the proportion of each stratum
    strata_counts = df.groupby(stratification_base).size()
    strata_proportions = strata_counts / len(df)
    
    # Calculate how many samples to take from each stratum
    # Use floor first, then distribute remaining samples to the largest remainders
    sample_sizes = (strata_proportions * num_entries).apply(np.floor).astype(int)
    remainder = num_entries - sample_sizes.sum()
    
    if remainder > 0:
        # Add the remaining samples to the strata with the largest fractional parts
        fractional_parts = (strata_proportions * num_entries) - sample_sizes
        extra_idx = fractional_parts.nlargest(remainder).index
        sample_sizes.loc[extra_idx] += 1
    
    # Sample from each stratum
    samples = []
    for stratum, size in sample_sizes.items():
        if size == 0:
            continue
        # Handle multi-column stratification
        if len(stratification_base) == 1:
            mask = df[stratification_base[0]] == stratum
        else:
            mask = True
            for col, val in zip(stratification_base, stratum if isinstance(stratum, tuple) else [stratum]):
                mask &= (df[col] == val)
        
        stratum_df = df[mask]
        # If the requested size is larger than available, take all
        size = min(size, len(stratum_df))
        samples.append(stratum_df.sample(n=size, replace=False))
    
    if not samples:
        return df.iloc[0:0].copy()  # empty DataFrame with same columns
    
    result = pd.concat(samples, axis=0)
    return result.sample(frac=1).reset_index(drop=True)  # shuffle the final result

In [ ]:
def sample_stratified_percentage(
    df: pd.DataFrame,
    coverage: float,
    stratification_base: str,
) -> pd.DataFrame:
    """
    Sample a fraction of the DataFrame while selecting
    almost equal numbers of rows from every group of
    the stratification column (e.g. chapter).

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame (e.g. chunks_df with columns 'chapter', 'contents').
    coverage : float
        Overall fraction of rows to keep (0 < coverage ≤ 1).
        Example: 0.3 → keep ~30 % of the total rows,
        distributed as evenly as possible across chapters.
    stratification_base : str
        Column used for stratification (e.g. "chapter").

    Returns
    -------
    pd.DataFrame
        New DataFrame containing the balanced sample.
    """
    if not 0.0 < coverage <= 1.0:
        raise ValueError("coverage must be a float in the interval (0, 1]")

    if stratification_base not in df.columns:
        raise ValueError(f"Column '{stratification_base}' not found in DataFrame")

    if df.empty:
        return df.copy()

    # Total number of rows we want to keep
    n_total = max(1, int(round(len(df) * coverage)))

    # Unique groups (chapters)
    groups = df[stratification_base].unique()
    n_groups = len(groups)

    if n_groups == 0:
        return df.copy()

    # Target number of rows per group (almost equal)
    base = n_total // n_groups
    remainder = n_total % n_groups          # some groups get +1

    sampled_parts = []

    for i, group_value in enumerate(groups):
        group_df = df[df[stratification_base] == group_value]

        # Desired size for this group
        target = base + (1 if i < remainder else 0)

        # Never ask for more rows than the group actually has
        n_to_sample = min(target, len(group_df))

        if n_to_sample > 0:
            sampled_parts.append(
                group_df.sample(n=n_to_sample, random_state=None)
            )

    if not sampled_parts:
        return df.iloc[0:0].copy()          # empty DataFrame with same columns

    result = pd.concat(sampled_parts, ignore_index=True)
    return result

In [ ]:
def sample_multiple(
    df: pd.DataFrame,
    num_chunks: int,
    num_entries: int,
    column: str,
    *,
    replace: bool = False
) -> pd.DataFrame:
    """
    Sample groups of rows from a DataFrame and return a summary DataFrame.

    For each group, the values of the specified column are joined with
    `entry_separator`.

    Parameters
    ----------
    df : pd.DataFrame
        Source DataFrame whose records (rows) will be sampled.
    num_chunks : int
        Number of rows to select in each group ("in one pass").
    num_entries : int
        Number of independent groups (entries) to generate.
    column : str
        Name of the column whose values will be joined.
    replace : bool, default False
        Whether to sample with replacement within a group.
        If False and num_chunks > len(df), a ValueError is raised.

    Returns
    -------
    pd.DataFrame
        DataFrame with two columns:
        - "num_chunks" : the input parameter (constant across rows)
        - "selected_entries" : string formed by joining the selected
          column values with `entry_separator`
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in DataFrame")
    if num_chunks < 1:
        raise ValueError("num_chunks must be a positive integer")
    if num_entries < 1:
        raise ValueError("num_entries must be a positive integer")
    if not replace and num_chunks > len(df):
        raise ValueError(
            f"num_chunks ({num_chunks}) cannot exceed the number of rows "
            f"in df ({len(df)}) when replace=False"
        )

    results = []
    for _ in range(num_entries):
        sampled = df.sample(n=num_chunks, replace=replace)
        values = sampled[column].astype(str).tolist()
        chapters = sampled["chapter"].astype(str).tolist()

        results.append(
            {
                "num_chunks": num_chunks,
                "chunks": values,
                "chapter": chapters
            }
        )

    return pd.DataFrame(results)

### Basic

In [ ]:
pdf_coverage = sample_stratified(
    df=chunks_df,
    num_entries=10,
    stratification_base="chapter",
)

In [ ]:
pdf_coverage['type'] = 'basic'

This section randomly selects simple chunks from the input corpus until the predefined coverage threshold is reached. The selected chunks are then sent to the LLM to generate a question.

### Information Integration & F9 – Multi-Hop Reasoning Gaps

In [ ]:
pdf_failure_modes.loc[9, 'Failure Mode'], pdf_failure_modes.loc[9, 'Description']

In [ ]:
pdf_info_integration = sample_multiple(
    chunks_df,
    num_chunks=2,
    num_entries=10,
    column='chunks'
)

In [ ]:
pdf_info_integration['type'] = 'information_integration'

In [ ]:
pdf_info_integration.loc[0, 'chunks']

### Negative – Unsupported questions
_Tests whether the RAG system recognizes when the knowledge base does not contain the answer, instead of hallucinating one._

In [ ]:
pdf_unsupported_questions = sample_stratified(
    chunks_df,
    num_entries=10,
    stratification_base='chunks'
)

In [ ]:
pdf_unsupported_questions['type'] = 'negative_unsupported_questions'

In [ ]:
pdf_unsupported_questions.loc[0, 'chunks']

### Complexity – Flesch score
_Tests whether the RAG system can understand difficult passages in the knowledge base._

In [ ]:
pdf_complexity_questions = sample_stratified(
    chunks_df,
    num_entries=10,
    stratification_base='chunks'
)

In [ ]:
pdf_complexity_questions['type'] = 'complexity'

In [ ]:
pdf_complexity_questions.loc[0, 'chunks']

## Merging

In [ ]:
pdf_all = pd.concat([
        pdf_coverage[['type', 'chapter', 'chunks']],
        pdf_info_integration[['type', 'chapter', 'chunks']],
        pdf_unsupported_questions[['type', 'chapter', 'chunks']],
        pdf_complexity_questions[['type', 'chapter', 'chunks']]
    ],
    ignore_index=True)

pdf_all['source'] = 'Laudon_MIS_17th.pdf'

In [ ]:
print('Coverage: {:.2f}%'.format(
    pdf_all.shape[0]/chunks_df.shape[0]*100
))

In [ ]:
pdf_all['type'].value_counts()

## Question generation

In [ ]:
import os
import json
import time
import yaml

import litellm
from pydantic import BaseModel

In [ ]:
llm_config_entry = 'claude_5_5'

with open("llm_config.yaml") as stream:
    try:
        config = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

    if llm_config_entry in config:
        LLM_MODEL = config[llm_config_entry]['provider']
        os.environ["ANTHROPIC_API_KEY"] = config[llm_config_entry]['api_key']
    else:
        print("No entry found in llm_config.yaml for `{:}`".format(llm_config_entry))

In [ ]:
def call_claude(message, model, max_tokens=1024):
    response = litellm.completion(
        model=model,
        max_tokens=max_tokens,
        messages=[
            {
                "role": "user",
                "content": message
            }
        ],
    )
    return response.choices[0].message.content

def parse_answer(a):
    for i in a:
        if i.type == 'text':
            data = json.loads(i.text[8:-4])
            
            return data.get("question"), data.get("answer")

class GeneratedQuestionAnswer(BaseModel):
    question: str
    answer: str

def call_claude_json(message, model, max_tokens=1024):
    start = time.perf_counter()

    response = litellm.completion(
        model=model,
        max_tokens=max_tokens,
        messages=[
            {
                "role": "user",
                "content": message
            }
        ],
        response_format=GeneratedQuestionAnswer,  # LiteLLM supports Pydantic models directly
    )

    elapsed_time = time.perf_counter() - start

    # Parse the structured JSON response
    parsed = GeneratedQuestionAnswer.model_validate_json(
        response.choices[0].message.content
    )

    return {
        'question': parsed.question,
        'answer': parsed.answer,
        'model': model,
        'input_token_count': response.usage.prompt_tokens,
        'output_token_count': response.usage.completion_tokens,
        'elapsed_time': elapsed_time,
    }

In [ ]:
# check if question makes no sense
GENERIC_PROMPT = """
You will be provided instruction on how to create a question. Generate a question and the answer to the question in JSON format, like this:

{{
    'question': <question generated from context>,
    'answer': <answer comes here>
}}

Instruction: {:}

Context: {:}
"""

question_prompts = {
    "basic": "Generate a question that can be answered with the provided context.",
    "information_integration": "In the provided context there are 2 passages separated by | character, generate a question that can be answered only with both passages.",
    "negative_unsupported_questions": """
You are an expert test-case generator for Retrieval-Augmented Generation (RAG) systems.

Your task is to create NEGATIVE TEST CASES of the type “Unsupported questions” (also called rejection-type or out-of-knowledge-base queries).

### Goal
Generate questions whose answers are **not** contained in the provided document(s).  
The correct expected behavior of a well-behaved RAG system is to refuse to answer and explicitly state that the information is insufficient (e.g., “Insufficient information to answer this question”, “I don’t know”, or an equivalent clear rejection).

These tests are intended to detect hallucinations: the system answering with plausible but unsupported content instead of acknowledging the knowledge gap.

### Input
You will receive chunks of context.  
Treat the entire provided content as the only knowledge base that the RAG system will have access to.

### Generation rules
1. The question must ask for information, facts, numbers, procedures, definitions, comparisons, or details that are **completely absent** from the given document(s).
2. The question should look realistic and natural — it can be thematically related to the document’s domain (so that retrieval might still pull some chunks) but the specific information requested must not be present.
3. Do **not** create questions that can be answered (even partially) from the document.
4. Prefer open-ended questions that would normally require a factual or explanatory answer (avoid pure yes/no questions).
5. Make the questions clear, unambiguous, and self-contained.
6. Vary the difficulty and style: some can be simple factual questions, others can request synthesis, specific numbers, causes, recommendations, etc., as long as the answer is unsupported.

### Two complementary strategies (use both)
- **Random / fabricated**: Invent plausible questions that simply have no corresponding information in the document.
- **Cross-document / topic-shift**: If multiple documents are provided, generate a question that would be answerable from one document but is asked while only another (unrelated) document is available. If only one document is given, create a question from a closely related but different topic.
""",
    "complexity": """You are an expert test-case generator for Retrieval-Augmented Generation (RAG) systems.

Your task is to create COMPLEXITY-ORIENTED TEST CASES that target the most difficult-to-read passages in a document, following the Flesch Reading Ease prioritization strategy used in RAG-TESTER.

### Goal
Generate questions based exclusively on the **most complex** passages of the provided document.  
Complex passages (low Flesch Reading Ease score) are harder for language models to parse, interpret, and synthesize. These tests are designed to expose failures related to comprehension deficits, incomplete extraction, or misinterpretation of dense, technical, or poorly readable text.

### Background on Flesch Reading Ease
- Score ranges from 0 to 100.
- Lower scores = higher complexity / more difficult to read.
- Typical interpretation:
  - 0–30  → Very difficult (academic, legal, scientific)
  - 30–50 → Difficult
  - 50–60 → Fairly difficult
  - 60–70 → Standard
  - 70+   → Easy

You should prefer (or be given) passages with the lowest Flesch scores.

### Input
You will receive one or more document chunks that have already been ranked by increasing Flesch Reading Ease score (most complex first).  
Each chunk should be used for at most one question.

### Generation rules
1. Create one question per provided complex chunk.
2. The question must be fully answerable from the content of that specific chunk (positive test).
3. Prefer questions that require careful reading, synthesis of dense information, interpretation of technical language, numbers, nested clauses, or subtle relationships present in the complex text.
4. Questions must be clear, unambiguous, and cannot be answered with a simple “yes” or “no”.
5. Do not invent information that is not present in the chunk.
6. The expected answer should be a faithful and complete extraction or synthesis of the relevant information from the chunk.

### Example
Complex chunk (low Flesch):  
“Notwithstanding the aforementioned provisions of subsection (b)(3)(A), the Secretary may, upon a finding that extraordinary circumstances exist which would render strict compliance with the requirements of this paragraph impracticable or contrary to the public interest, waive any such requirement for a period not to exceed 180 days…”

Good question:  
“Under what conditions and for how long may the Secretary waive the requirements of subsection (b)(3)(A)?”

Expected answer should accurately reflect the waiver conditions and the 180-day limit.

Now generate the requested number of high-quality complexity-oriented test cases based on the most difficult passages provided."""
}

In [ ]:
response = call_claude_json(
        GENERIC_PROMPT.format(
            question_prompts[pdf_all.iloc[0]['type']],
            pdf_all.iloc[0]['chunks']
        ),
        model = LLM_MODEL
    )

In [ ]:
response

### Batch

In [ ]:
from tqdm import tqdm

from Helper import Log

In [ ]:
pdf_all['question'] = ''
pdf_all['answer'] = ''
pdf_all['model'] = ''
pdf_all['input_token_count'] = 0
pdf_all['output_token_count'] = 0
pdf_all['elapsed_time'] = 0

for i in tqdm(range(pdf_all.shape[0])):
    if isinstance(pdf_all.iloc[i]['chunks'], list):
        chunks = '|'.join(pdf_all.iloc[i]['chunks'])
    else:
        chunks = pdf_all.iloc[i]['chunks']
    
    response = call_claude_json(
        GENERIC_PROMPT.format(
            question_prompts[pdf_all.iloc[i]['type']],
            chunks
        ),
        model = CLAUDE_LLM
    )

    pdf_all.loc[pdf_all.index[i], 'question'] = response.get('question', '')
    pdf_all.loc[pdf_all.index[i], 'answer']   = response.get('answer', '')
    pdf_all.loc[pdf_all.index[i], 'model']   = response.get('model', '')
    pdf_all.loc[pdf_all.index[i], 'input_token_count'] = response.get('input_token_count', 0)
    pdf_all.loc[pdf_all.index[i], 'output_token_count'] = response.get('output_token_count', 0)
    pdf_all.loc[pdf_all.index[i], 'elapsed_time'] = response.get('elapsed_time', 0)

In [ ]:
pdf_all[['type', 'source', 'chapter', 'question', 'chunks', 'answer', 'model',
       'input_token_count', 'output_token_count', 'elapsed_time']][:5]

In [ ]:
 with Log.save_file("output/qa_set_{:}.csv".format(pdf_all.loc[0, 'source'][:-4])) as filename:
    pdf_all[['type', 'source', 'chapter', 'question', 'chunks', 'answer', 'model',
       'input_token_count', 'output_token_count', 'elapsed_time']].to_csv(filename, index=False)

__Merged table of errors__
| Module / Stage | Error name (Paper 1) | Error name (Paper 2) | Detectable by sample questions? |
| --- | --- | --- | --- |
| Chunking | E1 Overchunking | (part of) F7 Chunking Boundary Errors | Yes – incomplete coverage; missing consecutive info; fragmented answers |
| Chunking | E2 Underchunking | (part of) F7 Chunking Boundary Errors | Yes – mixed topics dilute relevance; extraneous content confuses generator |
| Chunking | E3 Context Mismatch | (part of) F7 Chunking Boundary Errors | Yes – broken links (e.g., definition split from subject); retrieval misses key context |
| Retrieval | E4 Missed Retrieval | (part of) F11 Low Recall / Ranking Failures | Yes – relevant chunks absent; incomplete/fabricated answers or unnecessary abstention |
| Retrieval | E5 Low Relevance | (related to) F8 Domain Embedding Mismatch / F11 | Yes – loosely related chunks; vocabulary mismatch or unbalanced corpus |
| Retrieval | E6 Semantic Drift | (related to) F8 / F11 | Yes – keyword match but intent mismatch |
| Reranking | E7 Low Recall | (part of) F11 Low Recall / Ranking Failures | Yes – needed chunks retrieved but ranked too low / not passed forward |
| Reranking | E8 Low Precision | (part of) F11 Low Recall / Ranking Failures | Yes – irrelevant chunks ranked high and confuse generation |
| Generation | E9 Abstention Failure | (related to aspects of F13/F15) | Yes – answers incorrectly when should abstain (missing/ambiguous info) |
| Generation | E10 Fabricated Content | F13 Hallucination Despite Context | Yes – unverifiable claims not grounded in retrieved context |
| Generation | E11 Parametric Overreliance | (subtype of) F13 Hallucination Despite Context | Yes – uses internal knowledge instead of (or against) retrieved docs |
| Generation | E12 Incomplete Answer | F15 Incomplete / Partial Answers | Yes – correct but missing critical details available in context |
| Generation | E13 Misinterpretation | (related to) F13 / F14 | Yes – misuses or misrepresents retrieved content |
| Generation | E14 Contextual Misalignment | (related to) F13 / F16 Incorrect Specificity | Yes – factual but off-topic / does not address the query |
| Generation | E15 Chronological Inconsistency | — (unique to Paper 1) | Yes – wrong temporal order or confused timelines |
| Generation | E16 Numerical Error | — (unique to Paper 1) | Yes – incorrect calculations or numerical misrepresentation |
| Ingestion | — | F1 Outdated/Stale Data | Partially – outdated answers on time-sensitive queries; needs corpus timestamps |
| Ingestion | — | F2 Missing/Incomplete Documents | Yes – systematic gaps; answers absent for known domain topics |
| Ingestion | — | F3 Layout Parsing Errors | Partially – tables/figures/columns fail; inspect parsed text vs. source |
| Ingestion | — | F4 Multimodality Conversion Loss | Partially – image/diagram/chart info lost; needs multimodal test cases |
| Representation | — | F5 Tokenization Fragmentation | Harder – domain terms split; needs embedding inspection or recall metrics |
| Representation | — | F6 Embedding Drift / Model Mismatch | Harder – quality drop after model updates; needs monitoring over time |
| Retrieval | — | F8 Domain Embedding Mismatch | Yes – general embeddings fail on domain queries |
| Retrieval | — | F9 Multi-Hop Reasoning Gaps | Yes – multi-document queries fail while single-hop succeed |
| Retrieval | — | F10 Graph RAG Trade-offs | Yes (if graph RAG used) – structure vs. coverage issues |
| Retrieval | — | F12 Position-of-Gold Bias | Yes – gold docs in middle of context ignored; reorder tests |
| Generation | — | F14 Conflicting Info Unresolved | Yes – contradictory docs; model fails to note or resolve conflict |
| Generation | — | F16 Incorrect Specificity | Yes – too general/specific vs. query intent |
| Generation | — | F17 Wrong Output Format | Yes – schema/JSON/citation format broken despite correct content |
| Evaluation | — | F18 Metric Inadequacy | No (meta) – standard metrics miss faithfulness/utility |
| Evaluation | — | F19 Lack of Continuous Monitoring | No (ops) – silent degradation over time |
| Deployment | — | F20 Latency / Cost Bottlenecks | Partially – slow/expensive responses under load |
| Deployment | — | F21 Auditability Gaps | Harder – insufficient logging of which docs influenced output |
| Deployment | — | F22 Domain Portability Degradation | Yes – performance drop when moving domains |
| Deployment | — | F23 PII / Compliance Leaks | Partially – sensitive data appears in outputs |
| Deployment | — | F24 Prompt Sensitivity | Yes – small prompt changes break behavior |
| Deployment | — | F25 Authorization / Policy Failures (AuthZ) | Partially – access-control violations |
| Agentic orchestration | — | F26 Planning Failures | Yes (multi-step) – bad plans on agentic queries |
| Agentic orchestration | — | F27 Tool Selection / Execution Errors | Yes – wrong tool or failed execution |
| Agentic orchestration | — | F28 Context Memory Degradation | Yes (multi-turn) – lost context over steps |
| Agentic orchestration | — | F29 Multi-Agent Coordination Failures | Yes – agents fail to coordinate |
| Agentic orchestration | — | F30 Recursive Hallucination Cascades | Yes – hallucinations amplify across steps |
| Agentic orchestration | — | F31 Unbounded Cost / Latency Spirals | Partially – cost/latency explodes on recursive runs |
| Agentic orchestration | — | F32 Unsafe Reasoning Chains | Yes – unsafe intermediate reasoning |
| Agentic orchestration | — | F33 Attribution & Governance Gaps | Harder – poor attribution across multi-step chains |